In [0]:
# STEP 1: IMPORTS

from pyspark.sql.functions import col
import smtplib
from email.mime.text import MIMEText


# STEP 2: READ ALERT TABLE

alerts = spark.read.table("retail_store_sales.gold.store_inventory_alerts")


# STEP 3: FILTER CRITICAL ALERTS

critical_alerts = alerts.filter(
    (col("alert_level") == "OUT OF STOCK") |
    (col("alert_level") == "CRITICAL")
)

# STEP 4: CHECK IF ALERTS EXIST

count = critical_alerts.count()

if count == 0:
    message = " No critical inventory alerts today."
else:
    # Convert to pandas for email formatting (safe since small data)
    pdf = critical_alerts.toPandas()

    message = f"""
  INVENTORY ALERT REPORT  

Total Critical Alerts: {count}

----------------------------------------
{pdf.to_string(index=False)}
----------------------------------------

Please take necessary action immediately.
"""


# STEP 5: EMAIL CONFIGURATION

sender_email = "nallagangulaanusha3@gmail.com"
receiver_email = "anushanallagangula19@gmail.com"
app_password = "xoui lcyg tjba aaaf"  


# STEP 6: CREATE EMAIL

msg = MIMEText(message)
msg['Subject'] = f" Inventory Alert - {count} Critical Issues"
msg['From'] = sender_email
msg['To'] = receiver_email


# STEP 7: SEND EMAIL

try:
    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
        server.login(sender_email, app_password)
        server.send_message(msg)

    print(" Email alert sent successfully!")

except Exception as e:
    print(" Failed to send email:", str(e))


# STEP 8: LOG OUTPUT (OPTIONAL)

print("----- ALERT SUMMARY -----")
print(message)

 Email alert sent successfully!
----- ALERT SUMMARY -----

  INVENTORY ALERT REPORT  

Total Critical Alerts: 270

----------------------------------------
store_id       product_id  stock_level event_time               ingestion_ts                                                                          product_name        category  list_price                             store_name             store_city  alert_level  alert_priority  estimated_stock_value
 STR3244  TEC-PH-10001536            7 2014-10-28 2026-05-03 11:42:13.219957                                                  Spigen Samsung Galaxy S5 Case Wallet      Technology       16.99                  Chicago Central Store                Chicago     CRITICAL               2                 118.93
 STR1534  OFF-SU-10003629            7 2011-08-04 2026-05-03 11:42:13.219957                                                      Fiskars Letter Opener, Easy Grip Office Supplies       18.62          Aix-en-provence Central Store     